In [1]:
import pandas as pd

orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')

print("Orders shape:", orders.shape)
print("Order items shape:", order_items.shape)

Orders shape: (99441, 8)
Order items shape: (112650, 7)


In [2]:
order_items['order_id'].value_counts().head(10)

order_id
8272b63d03f5f79c56e9e4120aec44ef    21
1b15974a0141d54e36626dca3fdc731a    20
ab14fdcfbe524636d65ee38360e22ce8    20
9ef13efd6949e4573a18964dd1bbe7f5    15
428a2f660dc84138d969ccd69a0ab6d5    15
9bdc4d4c71aa1de4606060929dee888c    14
73c8ab38f07dc94389065f7eba4f297a    14
37ee401157a3a0b28c9c6d0ed8c3b24b    13
2c2a19b5703863c908512d135aa6accc    12
c05d6a79e55da72ca780ce90364abed9    12
Name: count, dtype: int64

In [3]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')

print("Customers shape:", customers.shape)
customers.head()

Customers shape: (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [4]:
print("Unique customer_id values:", customers['customer_id'].nunique())
print("Unique customer_unique_id values:", customers['customer_unique_id'].nunique())

Unique customer_id values: 99441
Unique customer_unique_id values: 96096


In [5]:
order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

print("Order payments shape:", order_payments.shape)
order_payments.head()

Order payments shape: (103886, 5)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [6]:
order_payments['order_id'].value_counts().head(5)

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
fedcd9f7ccdc8cba3a18defedd1a5547    19
Name: count, dtype: int64

In [7]:
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

print("Products shape:", products.shape)
print("Sellers shape:", sellers.shape)
print("Order reviews shape:", order_reviews.shape)
print("Category translation shape:", category_translation.shape)

Products shape: (32951, 9)
Sellers shape: (3095, 4)
Order reviews shape: (99224, 7)
Category translation shape: (71, 2)


In [8]:
tables = {
    'orders': orders,
    'order_items': order_items,
    'customers': customers,
    'order_payments': order_payments,
    'products': products,
    'sellers': sellers,
    'order_reviews': order_reviews,
    'category_translation': category_translation
}

for name, df in tables.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"--- {name} ---")
    print(missing if len(missing) > 0 else "No missing values")
    print()

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- order_items ---
No missing values

--- customers ---
No missing values

--- order_payments ---
No missing values

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- sellers ---
No missing values

--- order_reviews ---
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- category_translation ---
No missing values



In [9]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')

print(products['product_category_name'].isnull().sum(), "missing values remaining")

0 missing values remaining


In [10]:
import sqlite3

conn = sqlite3.connect('../data/olist.db')
print("Database connection created")

Database connection created


In [11]:
orders.to_sql('orders', conn, if_exists='replace', index=False)
customers.to_sql('customers', conn, if_exists='replace', index=False)
order_items.to_sql('order_items', conn, if_exists='replace', index=False)
order_payments.to_sql('order_payments', conn, if_exists='replace', index=False)
order_reviews.to_sql('order_reviews', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
sellers.to_sql('sellers', conn, if_exists='replace', index=False)
category_translation.to_sql('category_translation', conn, if_exists='replace', index=False)

print("All tables loaded into database")

All tables loaded into database


In [12]:
query = "SELECT * FROM orders LIMIT 5"
result = pd.read_sql(query, conn)
result

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [13]:
query = """
SELECT 
    order_id,
    order_estimated_delivery_date,
    order_delivered_customer_date,
    julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) AS delivery_delay_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
LIMIT 10
"""
result = pd.read_sql(query, conn)
result

,order_id,order_estimated_delivery_date,order_delivered_customer_date,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-18 00:00:00,2017-10-10 21:25:13,-7.107488
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-13 00:00:00,2018-08-07 15:27:45,-5.355729
2,47770eb9100c2d0c44946d9cf07ec65d,2018-09-04 00:00:00,2018-08-17 18:06:29,-17.245498
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-15 00:00:00,2017-12-02 00:28:42,-12.980069
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-26 00:00:00,2018-02-16 18:17:02,-9.238171
5,a4591c265e18cb1dcee52889e2d8acc3,2017-08-01 00:00:00,2017-07-26 10:57:55,-5.543113
6,6514b8ad8028c9f2cc2374ded245783f,2017-06-07 00:00:00,2017-05-26 12:55:51,-11.461215
7,76c6e866289321a7c93b82b54852dc33,2017-03-06 00:00:00,2017-02-02 14:08:10,-31.410995
8,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-23 00:00:00,2017-08-16 17:14:30,-6.281597
9,e6ce16cb79ec1d90b1da9085a6118aeb,2017-06-07 00:00:00,2017-05-29 11:18:31,-8.528808


In [14]:
query = """
SELECT 
    COUNT(*) AS total_delivered_orders,
    SUM(CASE WHEN julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) > 0 THEN 1 ELSE 0 END) AS late_orders,
    ROUND(100.0 * SUM(CASE WHEN julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_percentage
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
"""
result = pd.read_sql(query, conn)
result

,total_delivered_orders,late_orders,late_percentage
0,96476,7827,8.11


In [15]:
query = """
SELECT 
    c.customer_state,
    COUNT(*) AS total_delivered_orders,
    SUM(CASE WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0 THEN 1 ELSE 0 END) AS late_orders,
    ROUND(100.0 * SUM(CASE WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_percentage
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY late_percentage DESC
"""
result = pd.read_sql(query, conn)
result

,customer_state,total_delivered_orders,late_orders,late_percentage
0,AL,397,95,23.93
1,MA,717,141,19.67
2,PI,476,76,15.97
3,CE,1279,196,15.32
4,SE,335,51,15.22
5,BA,3256,457,14.04
6,RJ,12353,1664,13.47
7,TO,274,35,12.77
8,PA,946,117,12.37
9,ES,1995,244,12.23


In [16]:
query = """
SELECT 
    ct.product_category_name_english,
    COUNT(*) AS total_reviews,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM order_reviews r
JOIN order_items oi ON r.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
GROUP BY ct.product_category_name_english
HAVING total_reviews >= 30
ORDER BY avg_review_score ASC
LIMIT 15
"""
result = pd.read_sql(query, conn)
result

,product_category_name_english,total_reviews,avg_review_score
0,diapers_and_hygiene,39,3.26
1,office_furniture,1687,3.49
2,fashion_male_clothing,131,3.64
3,fixed_telephony,262,3.68
4,party_supplies,43,3.77
5,fashio_female_clothing,50,3.78
6,furniture_mattress_and_upholstery,38,3.82
7,audio,361,3.83
8,home_confort,435,3.83
9,construction_tools_safety,193,3.84


In [17]:
query = """
SELECT 
    c.customer_state,
    COUNT(DISTINCT c.customer_unique_id) AS total_customers,
    ROUND(SUM(p.payment_value), 2) AS total_revenue,
    ROUND(SUM(p.payment_value) / COUNT(DISTINCT c.customer_unique_id), 2) AS avg_revenue_per_customer
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_payments p ON o.order_id = p.order_id
GROUP BY c.customer_state
ORDER BY avg_revenue_per_customer DESC
LIMIT 15
"""
result = pd.read_sql(query, conn)
result

,customer_state,total_customers,total_revenue,avg_revenue_per_customer
0,PB,519,141545.72,272.73
1,AC,77,19680.62,255.59
2,RO,240,60866.20,253.61
3,AP,67,16262.80,242.73
4,AL,401,96962.06,241.80
5,PA,949,218295.85,230.03
6,TO,273,61485.33,225.22
7,PI,482,108523.97,225.15
8,RR,45,10064.62,223.66
9,SE,342,75246.25,220.02


In [18]:
# 1. Delivery delay by state
query1 = """
SELECT 
    c.customer_state,
    COUNT(*) AS total_delivered_orders,
    SUM(CASE WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0 THEN 1 ELSE 0 END) AS late_orders,
    ROUND(100.0 * SUM(CASE WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_percentage
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY late_percentage DESC
"""
delivery_by_state = pd.read_sql(query1, conn)
delivery_by_state.to_csv('../data/delivery_delay_by_state.csv', index=False)

# 2. Review score by category
query2 = """
SELECT 
    ct.product_category_name_english,
    COUNT(*) AS total_reviews,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM order_reviews r
JOIN order_items oi ON r.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
GROUP BY ct.product_category_name_english
HAVING total_reviews >= 30
ORDER BY avg_review_score ASC
"""
reviews_by_category = pd.read_sql(query2, conn)
reviews_by_category.to_csv('../data/reviews_by_category.csv', index=False)

# 3. Customer value by region
query3 = """
SELECT 
    c.customer_state,
    COUNT(DISTINCT c.customer_unique_id) AS total_customers,
    ROUND(SUM(p.payment_value), 2) AS total_revenue,
    ROUND(SUM(p.payment_value) / COUNT(DISTINCT c.customer_unique_id), 2) AS avg_revenue_per_customer
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_payments p ON o.order_id = p.order_id
GROUP BY c.customer_state
ORDER BY avg_revenue_per_customer DESC
"""
customer_value_by_state = pd.read_sql(query3, conn)
customer_value_by_state.to_csv('../data/customer_value_by_state.csv', index=False)

print("All 3 CSVs exported")

All 3 CSVs exported
